# Episode 25: AI agent: a graph database and a calculator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep025-graph-agent.ipynb)

Our agent gets a fourth role, which queries smollerGraph in Cypher, so it can take two facts from a graph database and combine them with the calculator, and we measure what it kept of its old tasks.

[The video](https://youtu.be/MeQ2iTAjobQ) · [The episode's page](https://mvgpt.nenuadrian.com/ep025-graph-agent/) · [The docs](https://nenuadrian.github.io/mvGPT/graph-agent/) · [The code at `ep025/end`](https://github.com/nenuadrian/mvGPT/tree/ep025/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep025/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("graph")

## smollerGraph
The graph the agent asks, as in episode 24: smollerGraph at its episode 10, loaded with the
mammals and birds on Wikidata, then `sg serve` on port 8474, where mvGPT looks for it.

In [ ]:
if not os.path.isdir("/content/smollerGraph"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep010/end https://github.com/nenuadrian/smollerGraph /content/smollerGraph
!pip install --quiet cmake ninja
!cmake -S /content/smollerGraph -B /content/smollerGraph/build -G Ninja > /dev/null && cmake --build /content/smollerGraph/build --target sg -- --quiet
!rm -f /content/smollerGraph/wikidata.sg && /content/smollerGraph/build/sg load /content/smollerGraph/wikidata.sg /content/smollerGraph/data/wikidata.txt
sg = start("/content/smollerGraph/build/sg serve /content/smollerGraph/wikidata.sg 8474", "http://127.0.0.1:8474/api/names?prefix=R")

## The tests
The graph role and what it hands back, and each new task replayed through the agent, with the graph faked.

In [ ]:
!python -m pytest -q tests/test_agent.py tests/test_graph_agent.py

## The data
Episode 21's tasks, which the agent keeps (they need FineWeb-Edu's last file for the web
pages), episode 24's questions, whose held-out sets the evaluation asks the graph role, then
today's 45,000 examples into `data/graph-agent/`.

In [ ]:
!python -c "from data import WEB_SHARDS, download_shard; download_shard(WEB_SHARDS - 1)"  # FineWeb-Edu's last file, which pretraining never read
!python agent_data.py
!python graph_data.py /content/smollerGraph/data/wikidata.txt
!python graph_agent_data.py /content/smollerGraph/data/wikidata.txt

## Fine-tuning
Episode 24's model, from the Hub, fine-tuned for thirty minutes into the agent with a graph
role, to `checkpoints/graph-agent`. The episode's run on a laptop took 1,675 steps.

In [ ]:
!python graph_agent_sft.py 30

## Measuring it
The new tasks, the graph role on episode 24's questions, and episode 21's tasks. Without
those episodes' results here, the columns for them show a dash.

In [ ]:
!python graph_agent_eval.py checkpoints/graph-agent/model.pt

## Asking it
Two birds training never names, a weight in kilograms, a weight Wikidata hasn't got, a fact
saved to a file, the Lion and the Tiger, and the Okapi, whose name the graph role copies
wrong.

In [ ]:
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the King Penguin than the Great White Pelican?"
!python agent.py checkpoints/graph-agent/model.pt "Are the King Penguin and the African Penguin in the same family?"
!python agent.py checkpoints/graph-agent/model.pt "How many kilograms does the King Penguin weigh?"
!python agent.py checkpoints/graph-agent/model.pt "How much do the King Penguin and the African Penguin weigh together?"
!python agent.py checkpoints/graph-agent/model.pt "What family is the African Penguin in? Save the answer in notes.txt."
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the Lion than the Tiger?"
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the Okapi than the Baikal Seal?"

## In a chat
Episode 21's Gradio app with the new model, through Colab's proxy: each subagent shows as a step.

In [ ]:
from google.colab import output

app = start("python agent_app.py checkpoints/graph-agent/model.pt", "http://127.0.0.1:7861")
output.serve_kernel_port_as_iframe(7861, height=720)

Next: [episode 26, Constrained decoding: structured LLM output](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep026-constrained-decoding.ipynb)